# GAN Urban Design v2 — Tez Görselleri (Veriye-Bağlı)

Bu notebook, Drive'daki sample dosyalarını ve metrics.json'ları okuyarak tezde kullanılacak **veriye-bağlı görselleri** üretir:

- **Şekil 4.1** — Dataset örnekleri (uydu | sketch | map)
- **Şekil 4.2** — Sentetik sketch üretim adımları (Canny vs XDoG vs birleşim)
- **Şekil 4.7** — Niteliksel model karşılaştırması (girdi | hedef | Pix2Pix | CycleGAN | Enhanced)
- **Şekil 4.8** — Eğitim örnek progresi (epoch 25 → 100 → 150)

Çıktılar `figures/data/` altına PNG olarak yazılır + Drive'a yedeklenir.

**Önkoşul:** `colab_external_models.ipynb` Hücre 15'inden sonra çalıştırılmalı (Geliştirilmiş Pix2Pix sample dosyaları üretilmiş olmalı).

## 1) Kurulum

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, pathlib
REPO_DIR = '/content/gan-urban-design-v2'
if not os.path.isdir(REPO_DIR):
    os.chdir('/content')
    subprocess.run(['git', 'clone', 'https://github.com/ilker-23/gan-urban-design-v2.git'], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=False)
os.chdir(REPO_DIR)

!pip install -q matplotlib opencv-python-headless

FIG_DIR = pathlib.Path(REPO_DIR) / 'figures' / 'data'
FIG_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_FIG = pathlib.Path('/content/drive/MyDrive/thesis_runs/figures')
DRIVE_FIG.mkdir(parents=True, exist_ok=True)
print('Hazır:', FIG_DIR)

## 2) Şekil 4.1 — Dataset örnekleri
Birkaç eşli görseli yan yana göster: solda **uydu (A_sat)**, ortada **sentetik kroki (A_sketch)**, sağda **renkli map (B)**.

In [ ]:
import cv2, random, matplotlib.pyplot as plt
import numpy as np

# Veri yoksa otomatik üret
if not os.path.isdir('data/processed/maps_sketch/val/A'):
    print('>> Veri eksik, üretiliyor (~3-5 dk)...')
    if not os.path.isdir('data/maps/train'):
        subprocess.run(['bash', 'scripts/download_data.sh', 'maps'], check=True)
    subprocess.run([sys.executable, 'scripts/prepare_sketches.py',
                    '--input', 'data/maps', '--output', 'data/processed',
                    '--method', 'canny_xdog'], check=True)

sat_dir   = 'data/processed/maps_paired/val/A'   # uydu
sketch_dir= 'data/processed/maps_sketch/val/A'    # sentetik kroki
map_dir   = 'data/processed/maps_sketch/val/B'    # renkli map (hedef)

random.seed(42)
names = sorted(os.listdir(sketch_dir))
samples = random.sample(names, 4)

fig, axes = plt.subplots(4, 3, figsize=(11, 14))
for r, n in enumerate(samples):
    triplet = []
    for d, title in [(sat_dir, 'Uydu (A)'), (sketch_dir, 'Sentetik kroki'), (map_dir, 'Renkli plan (B)')]:
        p = pathlib.Path(d) / n
        if not p.exists():
            triplet.append(None); continue
        img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
        triplet.append(img)
    for c, (img, title) in enumerate(zip(triplet, ['Uydu (A)', 'Sentetik kroki', 'Renkli plan (B)'])):
        if img is None:
            axes[r,c].text(0.5, 0.5, '(yok)', ha='center'); axes[r,c].axis('off'); continue
        axes[r,c].imshow(img)
        if r == 0:
            axes[r,c].set_title(title, fontsize=12, fontweight='bold')
        axes[r,c].axis('off')

fig.suptitle('Şekil 4.1 — maps Veri Kümesinden Üçlü Örnekler (uydu / sentetik kroki / renkli plan)',
             fontsize=13, fontweight='bold', y=0.995)
plt.tight_layout()
OUT = FIG_DIR / 'fig_4_1_dataset_examples.png'
plt.savefig(OUT, dpi=200, bbox_inches='tight', facecolor='white')
subprocess.run(['cp', str(OUT), str(DRIVE_FIG)])
plt.show()
print('Kaydedildi:', OUT)

## 3) Şekil 4.2 — Sentetik sketch üretim adımları
Aynı renkli map görüntüsünden **Canny**, **XDoG** ve **birleşim** çıktıları.

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location('prep', 'scripts/prepare_sketches.py')
prep = importlib.util.module_from_spec(spec); spec.loader.exec_module(prep)

name = samples[0]   # ilk örnek
color = cv2.imread(f'data/processed/maps_sketch/val/B/{name}')

canny = prep.make_sketch(color, method='canny')
xdog  = prep.make_sketch(color, method='xdog')
comb  = prep.make_sketch(color, method='canny_xdog')

fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
for ax, img, t in zip(axes,
                       [color, canny, xdog, comb],
                       ['Renkli plan (girdi)', 'Canny kenar', 'XDoG', 'Birleşim (kullanılan)']):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(t, fontsize=11, fontweight='bold')
    ax.axis('off')
fig.suptitle('Şekil 4.2 — Sentetik Sketch Üretiminin Üç Yöntemi (Eşitlik 3.1–3.4)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
OUT = FIG_DIR / 'fig_4_2_sketch_methods.png'
plt.savefig(OUT, dpi=200, bbox_inches='tight', facecolor='white')
subprocess.run(['cp', str(OUT), str(DRIVE_FIG)])
plt.show()
print('Kaydedildi:', OUT)

## 4) Şekil 4.7 — Niteliksel model karşılaştırması
Aynı sketch girdileri üzerinde **üç modelin çıktıları** + gerçek hedef yan yana.

Önkoşul: Pix2Pix baseline + Enhanced sample'ları Drive'da var. CycleGAN sample'ları external repo results klasöründe.

In [ ]:
# === Sekil 4.7 - Uc modelin niteliksel karsilastirmasi ===
# CycleGAN: external/ klonlanmadiysa Drive'daki yedek klasorden okur
# (colab_external_models.ipynb Hucre 9'da yedek alinmis olmali)
import torch, json, glob, numpy as np
from PIL import Image

PIX_RUN = '/content/drive/MyDrive/thesis_runs/pix2pix_sketch2map'
ENH_RUN = '/content/drive/MyDrive/thesis_runs/pix2pix_enhanced'

# CycleGAN icin uc olasi kaynak: external/ canli, Drive yedek (eval), Drive yedek (results)
CYC_CANDIDATES = [
    'external/pytorch-CycleGAN-and-pix2pix/results/sketch2map_cyc/test_latest/images',
    '/content/drive/MyDrive/thesis_runs/cyclegan_sketch2map/eval/fake',
    '/content/drive/MyDrive/thesis_runs/cyclegan_sketch2map/results/sketch2map_cyc/test_latest/images',
]
CYC_DIR = None
CYC_MODE = None  # 'junyanz' (real_A/fake_B/real_B) veya 'eval_dump' (00001.png basit)
for cand in CYC_CANDIDATES:
    if os.path.isdir(cand) and len(os.listdir(cand)) > 0:
        CYC_DIR = cand
        sample_files = os.listdir(cand)
        if any('_fake_B.' in f for f in sample_files):
            CYC_MODE = 'junyanz'
        else:
            CYC_MODE = 'eval_dump'
        break
print('CycleGAN kaynagi:', CYC_DIR, '| mod:', CYC_MODE)

# Drive'daki eval dump'inda eslesen real klasoru da var
CYC_REAL_DIR = None
if CYC_MODE == 'eval_dump' and CYC_DIR:
    candidate = CYC_DIR.replace('/fake', '/real')
    if os.path.isdir(candidate):
        CYC_REAL_DIR = candidate
        print('CycleGAN gercek klasoru:', CYC_REAL_DIR)

def best_epoch_dir(run_dir):
    eps = sorted(pathlib.Path(run_dir).glob('samples/epoch_*'))
    return eps[-1] if eps else None

pix_dir = best_epoch_dir(PIX_RUN)
enh_dir = best_epoch_dir(ENH_RUN)
print('Pix2Pix son epoch:', pix_dir)
print('Enhanced son epoch:', enh_dir)

def split_triplet(panel_path):
    if not panel_path or not panel_path.exists():
        return None, None, None
    img = cv2.imread(str(panel_path))
    if img is None:
        return None, None, None
    H, W = img.shape[:2]
    w = W // 3
    return img[:, :w], img[:, w:2*w], img[:, 2*w:]

# 4 ornek panel sec
panels = sorted(pix_dir.glob('sample_*.png'))[:4] if pix_dir else []
assert panels, f'Pix2Pix sample yok: {pix_dir}'

# CycleGAN cikti dosyalarini sirala
if CYC_DIR:
    if CYC_MODE == 'junyanz':
        cyc_files = sorted(glob.glob(f'{CYC_DIR}/*_fake_B.png'))
    else:
        cyc_files = sorted(glob.glob(f'{CYC_DIR}/*.png'))
    print(f'CycleGAN cikti dosyasi: {len(cyc_files)}')
else:
    cyc_files = []

fig, axes = plt.subplots(len(panels), 5, figsize=(18, len(panels)*3.5))
if len(panels) == 1:
    axes = axes[None, :]
col_titles = ['Girdi (kroki)', 'Hedef (gercek)', 'Pix2Pix (256)', 'CycleGAN (256)', 'Enhanced (512+VGG)']

for r, panel in enumerate(panels):
    A, B_real, B_pix = split_triplet(panel)
    B_enh = split_triplet(enh_dir / panel.name)[2] if enh_dir else None

    # CycleGAN icin r. ornek
    B_cyc = None
    if r < len(cyc_files):
        B_cyc = cv2.imread(cyc_files[r])
        # Drive eval dump'inda fake/00001.png var; eslesen real ayri klasorde
        if CYC_MODE == 'eval_dump' and CYC_REAL_DIR:
            real_path = os.path.join(CYC_REAL_DIR, os.path.basename(cyc_files[r]))
            # Hedef her durumda Pix2Pix panelinden geliyor zaten (ayni val seti)

    for c, img in enumerate([A, B_real, B_pix, B_cyc, B_enh]):
        ax = axes[r, c]
        if img is None:
            ax.text(0.5, 0.5, '(yok)', ha='center', va='center', fontsize=10, color='gray')
        else:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        if r == 0:
            ax.set_title(col_titles[c], fontsize=12, fontweight='bold')
        ax.axis('off')

fig.suptitle('Sekil 4.7 - Uc Modelin Niteliksel Karsilastirmasi (val kumesinden ornekler)',
             fontsize=14, fontweight='bold', y=1.005)
plt.tight_layout()
OUT = FIG_DIR / 'fig_4_7_qualitative.png'
plt.savefig(OUT, dpi=200, bbox_inches='tight', facecolor='white')
subprocess.run(['cp', str(OUT), str(DRIVE_FIG)])
plt.show()
print('Kaydedildi:', OUT)


## 5) Şekil 4.8 — Eğitim epoch progresi (Enhanced)
Aynı val örneği için epoch 25 → 75 → 150 görünümü: modelin nasıl öğrendiği.

In [ ]:
# === Sekil 4.8 - Egitim asamasina gore cikti kalitesi (akilli ornek secimi) ===
# Tum sample panellerden gercege en yakin olani (SSIM ile) secer
from skimage.metrics import structural_similarity as ssim_fn

RUN = pathlib.Path(ENH_RUN)
ep_dirs = sorted(RUN.glob('samples/epoch_*'))
print('Mevcut epoch sample dizinleri:', [d.name for d in ep_dirs])
if not ep_dirs:
    raise RuntimeError(f'{RUN}/samples altinda hic epoch klasoru yok')

# 3 farkli epoch sec (baslangic, orta, son)
if len(ep_dirs) >= 3:
    sel = [ep_dirs[0], ep_dirs[len(ep_dirs)//2], ep_dirs[-1]]
else:
    sel = ep_dirs

# SON epoch'taki tum panellerden, model ciktisi hedefe en yakin olani sec
last = sel[-1]
candidates = sorted(last.glob('sample_*.png'))
print(f'{last.name}: {len(candidates)} aday panel taraniyor...')
scores = []
for p in candidates:
    A, B_real, B_fake = split_triplet(p)
    if B_real is None or B_fake is None:
        scores.append(-1); continue
    if B_real.shape != B_fake.shape:
        B_fake = cv2.resize(B_fake, (B_real.shape[1], B_real.shape[0]))
    real_rgb = cv2.cvtColor(B_real, cv2.COLOR_BGR2RGB)
    fake_rgb = cv2.cvtColor(B_fake, cv2.COLOR_BGR2RGB)
    s = ssim_fn(real_rgb, fake_rgb, channel_axis=-1, data_range=255)
    scores.append(s)
best_idx = int(np.argmax(scores))
print(f'En iyi panel: {candidates[best_idx].name} (SSIM={scores[best_idx]:.4f})')
panel_name = candidates[best_idx].name

fig, axes = plt.subplots(len(sel), 3, figsize=(13, len(sel)*4.0))
if len(sel) == 1:
    axes = axes[None, :]
for r, ed in enumerate(sel):
    A, B_real, B_fake = split_triplet(ed / panel_name)
    for c, img in enumerate([A, B_fake, B_real]):
        ax = axes[r, c]
        if img is not None:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        if r == 0:
            titles = ['Girdi (kroki)', 'Model ciktisi', 'Hedef (gercek)']
            ax.set_title(titles[c], fontsize=12, fontweight='bold')
        if c == 0:
            label = ed.name.replace('_', ' ')
            ax.text(-0.1, 0.5, label, rotation=90, transform=ax.transAxes,
                    ha='center', va='center', fontsize=11, fontweight='bold')
        ax.set_xticks([]); ax.set_yticks([])

fig.suptitle('Sekil 4.8 - Gelistirilmis Pix2Pix Egitim Asamasina Gore Cikti Kalitesi',
             fontsize=14, fontweight='bold', y=1.005)
plt.tight_layout()
OUT = FIG_DIR / 'fig_4_8_training_progress.png'
plt.savefig(OUT, dpi=200, bbox_inches='tight', facecolor='white')
subprocess.run(['cp', str(OUT), str(DRIVE_FIG)])
plt.show()
print('Kaydedildi:', OUT)


## 6) Tüm görselleri repo'ya commit etmek
Aşağıdaki hücre, üretilen veri-bağlı görselleri GitHub repo'sunda `figures/data/` altına commit eder (PAT gerekirse hücre içi `getpass` ile).

In [ ]:
# (opsiyonel) — Görselleri Drive üzerinden Word'e ekleyebilirsiniz; commit'e gerek yok.
# Drive yolu:
!ls /content/drive/MyDrive/thesis_runs/figures